# 03 - MONAI 2D U-Net baseline

This notebook trains a binary LV segmentation baseline on processed EchoNet frame/mask pairs. Run `02_create_masks.ipynb` first so `data/processed/images/` and `data/processed/masks/` exist.

In [ ]:
# Kaggle execution order:
# 1. Run 01_explore_dataset.ipynb to validate raw files.
# 2. Run 02_create_masks.ipynb to create processed pairs.
# 3. Run this notebook with RUN_MODE = 'smoke' for one epoch.
# 4. Switch RUN_MODE = 'full' for the real experiment.
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import torch
from monai.data import DataLoader, Dataset

from src.dataset import get_monai_transforms, load_processed_samples, split_by_echonet_filelist, split_samples, SplitConfig
from src.model import build_unet
from src.train import evaluate, fit, get_loss, plot_training_curves, save_prediction_examples
from src.utils import load_echonet_tables, set_seed

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "EchoNet-Dynamic"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CHECKPOINT_DIR = PROJECT_ROOT / "outputs" / "checkpoints"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

set_seed(42)

## Configuration

`smoke` is a one-epoch end-to-end verification run for Kaggle. `full` is the baseline experiment template for 50+ epochs.

In [ ]:
RUN_MODE = 'smoke'  # change to 'full' for full training

SMOKE_CONFIG = {
    'epochs': 1,
    'batch_size': 8,
    'learning_rate': 1e-3,
    'max_train_samples': 32,
    'max_val_samples': 16,
    'max_test_samples': 16,
}

FULL_CONFIG = {
    'epochs': 50,
    'batch_size': 32,
    'learning_rate': 1e-3,
    'max_train_samples': None,
    'max_val_samples': None,
    'max_test_samples': None,
}

config = SMOKE_CONFIG if RUN_MODE == 'smoke' else FULL_CONFIG
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Run mode: {RUN_MODE}")
print(f"Device: {device}")
config

## Load processed pairs and create splits

The preferred split uses EchoNet's official `Split` column. If a partial preprocessing run does not contain enough matched samples, the notebook falls back to a reproducible random split.

In [ ]:
file_list, _ = load_echonet_tables(RAW_DIR)
samples = load_processed_samples(PROCESSED_DIR)
print(f"Processed image-mask pairs: {len(samples):,}")

train_samples, val_samples, test_samples = split_by_echonet_filelist(samples, file_list)
if min(len(train_samples), len(val_samples), len(test_samples)) == 0:
    train_samples, val_samples, test_samples = split_samples(samples, SplitConfig(seed=42))

train_samples = train_samples[:config['max_train_samples']]
val_samples = val_samples[:config['max_val_samples']]
test_samples = test_samples[:config['max_test_samples']]

print(f"Train: {len(train_samples):,} | Val: {len(val_samples):,} | Test: {len(test_samples):,}")

In [ ]:
train_ds = Dataset(data=train_samples, transform=get_monai_transforms(image_size=(112, 112), augment=True))
val_ds = Dataset(data=val_samples, transform=get_monai_transforms(image_size=(112, 112), augment=False))
test_ds = Dataset(data=test_samples, transform=get_monai_transforms(image_size=(112, 112), augment=False))

train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=config['batch_size'], shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_ds, batch_size=config['batch_size'], shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())

## Train U-Net baseline

The network predicts one logit channel. `DiceCELoss(sigmoid=True)` combines Dice overlap with BCE-style stabilization, which is useful for small foreground structures.

In [ ]:
model = build_unet(
    spatial_dims=2,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
).to(device)

loss_fn = get_loss('dice_bce')
optimizer = torch.optim.AdamW(model.parameters(), lr=config['learning_rate'], weight_decay=1e-5)

history = fit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    device=device,
    epochs=config['epochs'],
    checkpoint_dir=CHECKPOINT_DIR,
)

plot_training_curves(history, FIGURES_DIR / f'training_curves_{RUN_MODE}.png')
history

## Evaluate best checkpoint on held-out test set

This section reports final test Dice and saves representative prediction figures for later qualitative review and XAI setup.

In [ ]:
best_checkpoint = torch.load(CHECKPOINT_DIR / 'best_unet.pt', map_location=device)
model.load_state_dict(best_checkpoint['model_state_dict'])

test_metrics = evaluate(model, test_loader, loss_fn, device)
print(f"Final test loss: {test_metrics['loss']:.4f}")
print(f"Final test Dice: {test_metrics['dice']:.4f}")

save_prediction_examples(
    model=model,
    loader=test_loader,
    device=device,
    output_dir=FIGURES_DIR,
    max_examples=6 if RUN_MODE == 'full' else 3,
)
test_metrics